# Расчет относительных показателей по туберкулезу 2016-2024

Дата: 27.04.2026

**Заказчик:** ОГБУЗ «Иркутская областная клиническая туберкулезная больница». 

**Цель:** используя ранее созданный **indicators.jsonl** (формат jsonl) файл-справочник с формулами и таблицу с абсолютными числами из медицинских статистических форм №8 "Сведения о заболеваниях активным туберкулезом", №30 "Сведения о медицинской организации", №33 "Сведения о больных туберкулезом" рассчитать относительные показатели по годам.

Для расчета используются файлы:
- population.csv - среднегодовое население общее и в разрезе пола и возраста

 Структура:
 year, all, man, woman, all_18+, children_0_14, children_0_14_m, children_0_14_w, children_0_17, children_0_17_m, children_0_17_w, children_0_4,
 children_0_4_m, children_0_4_w, children_5_6, children_5_6_m, children_5_6_w, children_7_14, children_7_14_m, children_7_14_w, children_15_17,
 children_15_17_m, children_15_17_w, all_18_24, all_18_24_m, all_18_24_w, all_25_34, all_25_34_m, all_25_34_w, all_35_44, all_35_44_m, all_35_44_w,
 all_45_54, all_45_54_m, all_45_54_w, all_55_64, all_55_64_m, all_55_64_w, all_65+, all_65+_m, all_65+_w, all_15+, all_15+_m, all_15+_w, 
 
 - 8_33_30_2016-2024.csv - "длинный формат" файла с абсолютными значениями из медицинских статистических форм №8, № 30, № 33 по годам.

Структура:
  - Показатель
  - Уточнение
  - Еще уточнение
  - Возраст
  - Год
  - Значение
  - Строка
  - Графа
  - Таблица

In [1]:
import pandas as pd
import math
import json

In [2]:
# Настройки отображения DataFrame
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)

In [3]:
# путь к файлам
path = 'd:/datasets/tuberculosis/Свод/'

In [4]:
def load_data_from_list(file_name, folder=path):
    """
    Функция загружает файл из списка file_name из папки folder
    """
    full_path = f"{folder}{file_name}"
    try:
        df = pd.read_csv(full_path, encoding='utf-8', encoding_errors='ignore')
        display(f"Файл загружен успешно: {file_name} | Строк: {df.shape[0]}")
        return df
    except Exception as e:
        print(f"Ошибка при загрузке: {file_name}: {e}")
        return None

In [5]:
# список файлов для расчета - таблица с данными по годам и среднегодовое население по годам
list_file = ["8_33_30_2016-2024.csv", "population.csv"]

In [6]:
df = load_data_from_list(list_file[0],'d:/datasets/tuberculosis/Свод/')

'Файл загружен успешно: 8_33_30_2016-2024.csv | Строк: 9052'

In [7]:
df_population = load_data_from_list(list_file[1],'d:/datasets/tuberculosis/Свод/')

'Файл загружен успешно: population.csv | Строк: 9'

In [8]:
df_population.head(10) 

,year,all,man,woman,all_18+,children_0_14,children_0_14_m,children_0_14_w,children_0_17,children_0_17_m,children_0_17_w,children_0_4,children_0_4_m,children_0_4_w,children_5_6,children_5_6_m,children_5_6_w,children_7_14,children_7_14_m,children_7_14_w,children_15_17,children_15_17_m,children_15_17_w,all_18_24,all_18_24_m,all_18_24_w,all_25_34,all_25_34_m,all_25_34_w,all_35_44,all_35_44_m,all_35_44_w,all_45_54,all_45_54_m,all_45_54_w,all_55_64,all_55_64_m,all_55_64_w,all_65+,all_65+_m,all_65+_w,all_15+,all_15+_m,all_15+_w
0,2016,2410851,1114619,1296232,1847685,488187,250277,237910,563166,288921,274246,183794,94369,89425,69207,35325,33883,235186,120584,114603,74979,38644,36336,190183,94937,95247,409323,206341,202982,348260,166344,181917,291441,134731,156710,318713,133076,185638,289765,90271,199495,1922664,864342,1058322
1,2017,2406548,1112389,1294159,1835655,494280,253527,240753,570895,292939,277956,180095,92675,87420,71944,36680,35265,242241,124173,118068,76615,39412,37204,179732,89836,89896,400958,202508,198450,351781,168114,183667,286420,132457,153964,317414,132864,184550,299350,93673,205678,1912269,858862,1053407
2,2018,2400979,1109440,1291539,1824483,497183,255032,242152,576497,295655,280842,173758,89448,84310,74222,37978,36245,249204,127607,121597,79314,40624,38691,173837,87221,86617,387116,195507,191609,356380,170847,185534,284386,131440,152946,314232,131825,182408,308532,96947,211585,1903796,854409,1049388
3,2019,2394478,1106966,1287513,1815414,497434,255255,242179,579066,297040,282026,166162,85593,80570,73833,37950,35883,257439,131713,125727,81632,41786,39847,171343,86326,85017,372520,188429,184092,360286,173554,186732,285077,131723,153355,308254,129695,178559,317934,100200,217734,1897045,851711,1045334
4,2020,2383107,1102145,1280962,1804559,495875,254436,241439,578551,296766,281785,156989,80884,76105,72584,37333,35251,266302,136219,130083,82676,42330,40346,170497,86258,84240,357878,181466,176412,363064,175607,187457,287566,132744,154822,299733,126549,173184,325821,102757,223064,1887233,847709,1039524
5,2021,2369234,1092153,1277082,1800830,487016,249846,237170,568407,291693,276714,145400,74826,70574,70721,36411,34311,270895,138610,132286,81391,41847,39545,169214,86206,83008,331043,166696,164347,373903,180203,193700,294843,135627,159217,294570,124839,169731,337257,106891,230366,1882219,842307,1039912
6,2022,2353904,1081052,1272852,1790669,480722,246752,233971,563235,289412,273823,136960,70490,66470,67889,35016,32874,275874,141246,134628,82513,42661,39853,170290,87068,83222,300100,149618,150482,384840,184591,200249,302033,138137,163896,287228,122027,165201,346178,110199,235980,1873182,834300,1038882
7,2023,2337449,1072490,1264959,1772529,476624,244846,231778,564922,290393,274529,131475,67706,63770,64080,32984,31097,281069,144157,136912,88298,45547,42751,173439,88519,84920,278524,139139,139386,387437,185924,201513,305264,138910,166354,276137,117596,158541,351728,112011,239717,1860825,827644,1033181
8,2024,2330537,1068992,1261545,1767956,470864,241933,228931,562581,289139,273442,127771,65725,62046,61716,31704,30012,281377,144504,136873,91717,47206,44511,174343,89016,85327,269012,134817,134195,388659,186602,202057,307305,139659,167646,271701,115967,155734,356936,113792,243144,1859673,827059,1032614


In [9]:
df.head(10)

,Показатель,Уточнение,Еще уточнение,Код по МКБ-Х пересмотра,Возраст,Пол,Год,Значение,Строка,Графа,Таблица
0,Выявлено в текущем году,NaN,NaN,NaN,Всего,NaN,2016,16.0,1,3,2800
1,Выявлено в текущем году,NaN,NaN,NaN,Детей от 0 до 14 лет,NaN,2016,0.0,1,4,2800
2,Выявлено в текущем году,NaN,NaN,NaN,Подростков 15-17 лет,NaN,2016,0.0,1,5,2800
3,"Выявлено в текущем году, из них с МБТ+",NaN,NaN,NaN,Всего,NaN,2016,1.0,2,3,2800
4,"Выявлено в текущем году, из них с МБТ+",NaN,NaN,NaN,Детей от 0 до 14 лет,NaN,2016,0.0,2,4,2800
5,"Выявлено в текущем году, из них с МБТ+",NaN,NaN,NaN,Подростков 15-17 лет,NaN,2016,0.0,2,5,2800
6,Наблюдалось в отчетном году,NaN,NaN,NaN,Всего,NaN,2016,5.0,3,3,2800
7,Наблюдалось в отчетном году,NaN,NaN,NaN,Детей от 0 до 14 лет,NaN,2016,0.0,3,4,2800
8,Наблюдалось в отчетном году,NaN,NaN,NaN,Подростков 15-17 лет,NaN,2016,0.0,3,5,2800
9,Лечились в стационаре,NaN,NaN,NaN,Всего,NaN,2016,1.0,4,3,2800


In [10]:
# jsonl-файл с формулами расчета показателей
file_indicators = 'd:/datasets/tuberculosis/Свод/indicators.jsonl'

In [11]:
# jsonl-файл с формулами расчета показателей
file_indicators = 'd:/datasets/tuberculosis/Свод/indicators.jsonl'

In [12]:
def calculate_indicator(file_indicators, target_id, year):
    ''' расчет показателей 
        на входе -  file_indicators - json-файл с формулами числителя и знаменателя
                    target_id - id показателя
                    year - год, за который считать показатель
        функция возвращает:
        - полное наименование показателя, 
        - значение рассчитанного показателя, 
        - числитель и знаменатель для проверки 
    '''
    with open(file_indicators, "r", encoding="utf-8") as f:
        
        current_year = year
        last_year = year-1
        
        for line in f:
            # 1. Читаем строку и превращаем в словарь
            record = json.loads(line.strip())
            
            # 2. Ищем нужный нам ID
            if record.get("id") == target_id:
                name_indicator = record.get("name")
                short_name_indicator =  record.get("short_name") 
                
                # Читаем множитель      
                multiplier_local = record.get("multiplier", 1)
                
                              
                # Читаем значение числителя
                data_list = record.get("numerator", [])
                                
                total_numerator = 0
                
                # Проходим циклом по всем элементам числителя
                for item in data_list:
                    table = item['table'] 
                    row = item['row']
                    col = item['col']
                    
                    if item['year'] == 'current':
                        table_year = current_year
                    if item['year'] == 'last':
                        table_year = last_year
                    
                    value_year = df.loc[(df['Год'] == table_year) & (df['Таблица'] == table) & \
                                   (df['Строка'] == row) & (df['Графа'] == col), 'Значение']
                    
                    # Складываем значения числителя
                    if not value_year.empty:
                        if ',' in str(value_year.iloc[0]) or '.' in str(value_year.iloc[0]):
                            total_numerator += float(str(value_year.iloc[0]).replace(',', '.'))
                        else:
                            total_numerator += int(value_year.iloc[0])
                
                    total_denominator = 0 # знаменатель
              
                # Читаем значение знаменателя
                data_list = record.get("denominator", [])
                                
                if 'table' in data_list[0]:
                    # Если таблица, то может быть несколько таблиц и нужна сумма их значений
                    
                    # Проходим циклом по всем элементам знаменателя
                    for item in data_list:
                        table = item['table'] 
                        row = item['row']
                        col = item['col']

                        if item['year'] == 'current':
                            table_year = current_year
                        if item['year'] == 'last':
                            table_year = last_year
                        
                        value_year = df.loc[(df['Год'] == table_year) & (df['Таблица'] == table) & \
                                    (df['Строка'] == row) & (df['Графа'] == col), 'Значение']
                        
                        # Складываем значения
                        if not value_year.empty:
                            if ',' in str(value_year.iloc[0]) or '.' in str(value_year.iloc[0]):
                                total_denominator += float(str(value_year.iloc[0]).replace(',', '.'))
                            else:    
                                total_denominator += int(value_year.iloc[0])
                elif 'population' in data_list[0]:
                    # Если в знаменателе население, то просто вытаскиваем его из таблицы
                    for item in data_list:
                        column_population = item['population']
                        population_year = df_population.loc[df_population['year'] == year, column_population]   
                        total_denominator = int(population_year.iloc[0])
                                        
                # Считаем основной показатель
                if total_denominator != 0:
                    result = total_numerator/total_denominator
                               
                    # Умножаем на множитель               
                    if multiplier_local != 0:
                        result = result *  multiplier_local
                else: 
                    print('Деление на 0!')    
                    result = 0

                result = round(result, 2)
                return name_indicator, short_name_indicator, result, total_numerator, total_denominator  # Возвращаем данные, если нашли
                
    display("Показатель не найден")
    return None

In [13]:
# Список показателей для расчета
list_indicators = [ "id_1.1", "id_1.2", "id_1.3", 
                    "id_1.4","id_1.4.1","id_1.4.2",\
                    "id_1.5","id_1.5.1","id_1.5.2",\
                    "id_1.6","id_1.6.1","id_1.6.2",\
                    "id_1.7","id_1.7.1","id_1.7.2",\
                    "id_1.8","id_1.8.1","id_1.8.2",\
                    "id_1.9","id_1.9.1","id_1.9.2",\
                    "id_1.10","id_1.10.1","id_1.10.2",\
                    "id_1.11","id_1.11.1","id_1.11.2",\
                    "id_1.12","id_1.12.1","id_1.12.2",\
                    "id_1.13","id_1.13.1","id_1.13.2",\
                    "id_2.1.1", "id_2.1.2", "id_2.1.3", "id_2.1.4",\
                    "id_2.1.5", "id_2.1.6",\
                    "id_2.2.1", "id_2.2.2", "id_2.2.3", "id_2.2.4",\
                    "id_2.2.5", "id_2.2.6",\
                    "id_2.3.1", "id_2.3.2", "id_2.3.3", "id_2.3.4",\
                    "id_2.3.5", "id_2.3.6",\
                    "id_2.4.1", "id_2.4.2", "id_2.4.3", "id_2.4.4",\
                    "id_2.4.5", "id_2.4.6", "id_2.4.7", "id_2.4.8",\
                    "id_2.5.1", "id_2.5.2", "id_2.5.3", "id_2.5.4",\
                    "id_2.5.5", "id_2.5.6", "id_2.5.7", "id_2.5.8",\
                    "id_2.6.1", "id_2.6.2",\
                    "id_2.7.1", "id_2.7.2", "id_2.7.3", "id_2.7.4",\
                    "id_2.7.5", "id_2.7.6",\
                    "id_2.8.1", "id_2.8.2", "id_2.8.3", "id_2.8.4",\
                    "id_2.8.5", "id_2.8.6",\
                    "id_2.9.1", "id_2.9.2", "id_2.9.3", "id_2.9.4",\
                    "id_2.9.5", "id_2.9.6",\
                    "id_2.10.1", "id_2.10.2", "id_2.10.3", "id_2.10.4",\
                    "id_2.10.5", "id_2.10.6", "id_2.10.7", "id_2.10.8",\
                     "id_3.1", "id_3.2", "id_3.3",  "id_3.4",\
                    "id_3.5", "id_3.6", "id_3.7", "id_3.8",\
                    "id_3.9", "id_3.10",\
                    "id_4.1", "id_4.2", "id_4.3", "id_4.4",\
                    "id_4.5",\
                    "id_5.1", "id_5.2", "id_5.3","id_5.4", "id_5.5",\
                    "id_6.1", "id_6.2", "id_6.3", "id_6.4", \
                    "id_6.5", "id_6.6","id_6.7",\
                    "id_7.1", "id_7.2", "id_7.3", "id_7.4", "id_7.5",\
                    "id_8.1", "id_8.2", "id_8.3", "id_8.4", "id_8.5",\
                    "id_8.6", "id_8.7", "id_8.8", \
                    "id_9.1", "id_9.2", "id_9.3", "id_9.4"]

In [14]:
results_list = []

In [15]:
for i in range(0,len(list_indicators)):
    # идем по списку показателей
    print(list_indicators[i])
    for year in range(2016,2024+1):
        # идем по годам
        # вызов функции расчета
        name, short_name, ind, numerator, denominator = calculate_indicator(file_indicators, list_indicators[i], year)    
        # добавляем в список результат расчета
        results_list.append({'id': list_indicators[i], 'name': name, 'short_name': short_name, 'year': year, 'indicator': ind, \
                             'numerator': numerator, "denominator": denominator})

id_1.1
id_1.2
id_1.3
id_1.4
id_1.4.1
id_1.4.2
id_1.5
id_1.5.1
id_1.5.2
id_1.6
id_1.6.1
id_1.6.2
id_1.7
id_1.7.1
id_1.7.2
id_1.8
id_1.8.1
id_1.8.2
id_1.9
id_1.9.1
id_1.9.2
id_1.10
id_1.10.1
id_1.10.2
id_1.11
id_1.11.1
id_1.11.2
id_1.12
id_1.12.1
id_1.12.2
id_1.13
id_1.13.1
id_1.13.2
id_2.1.1
id_2.1.2
id_2.1.3
id_2.1.4
id_2.1.5
id_2.1.6
id_2.2.1
id_2.2.2
id_2.2.3
id_2.2.4
id_2.2.5
id_2.2.6
id_2.3.1
id_2.3.2
id_2.3.3
id_2.3.4
id_2.3.5
id_2.3.6
id_2.4.1
id_2.4.2
id_2.4.3
id_2.4.4
id_2.4.5
id_2.4.6
id_2.4.7
id_2.4.8
id_2.5.1
id_2.5.2
id_2.5.3
id_2.5.4
id_2.5.5
id_2.5.6
id_2.5.7
id_2.5.8
id_2.6.1
id_2.6.2
id_2.7.1
id_2.7.2
id_2.7.3
id_2.7.4
id_2.7.5
id_2.7.6
id_2.8.1
id_2.8.2
id_2.8.3
id_2.8.4
id_2.8.5
id_2.8.6
id_2.9.1
id_2.9.2
id_2.9.3
id_2.9.4
id_2.9.5
id_2.9.6
id_2.10.1
id_2.10.2
id_2.10.3
id_2.10.4
id_2.10.5
id_2.10.6
id_2.10.7
id_2.10.8
id_3.1
id_3.2
id_3.3
id_3.4
id_3.5
id_3.6
id_3.7
id_3.8
id_3.9
id_3.10
id_4.1
id_4.2
id_4.3
id_4.4
id_4.5
id_5.1
id_5.2
id_5.3
id_5.4
id_5.5
id_6.1
id_

In [16]:
# Сохраняем в датафрейм
df_results = pd.DataFrame(results_list)

In [17]:
df_results.head(6)

,id,name,short_name,year,indicator,numerator,denominator
0,id_1.1,Показатель первичной заболеваемости туберкулез...,ПЗ туб тер,2016,108.47,2615.0,2410851.0
1,id_1.1,Показатель первичной заболеваемости туберкулез...,ПЗ туб тер,2017,96.57,2324.0,2406548.0
2,id_1.1,Показатель первичной заболеваемости туберкулез...,ПЗ туб тер,2018,82.09,1971.0,2400979.0
3,id_1.1,Показатель первичной заболеваемости туберкулез...,ПЗ туб тер,2019,74.21,1777.0,2394478.0
4,id_1.1,Показатель первичной заболеваемости туберкулез...,ПЗ туб тер,2020,61.01,1454.0,2383107.0
5,id_1.1,Показатель первичной заболеваемости туберкулез...,ПЗ туб тер,2021,55.00,1303.0,2369234.0


In [18]:
# Годы в столбцах
# Разворачиваем таблицу
df_pivot = df_results.pivot(index=['id', 'name'],columns='year', values=['indicator','numerator','denominator'])

# Сбрасываем индекс, чтобы "Показатель" стал обычной колонкой
df_pivot = df_pivot.reset_index()

In [19]:
df_pivot.head(10)

id                                               name indicator                                                                 numerator                                                                 denominator                                                                                        
year                                                                    2016    2017    2018    2019    2020    2021    2022    2023    2024      2016    2017    2018    2019    2020    2021    2022    2023    2024        2016       2017       2018       2019       2020       2021       2022       2023       2024
0        id_1.1  Показатель первичной заболеваемости туберкулез...    108.47   96.57   82.09   74.21   61.01   55.00   59.43   52.66   47.59    2615.0  2324.0  1971.0  1777.0  1454.0  1303.0  1399.0  1231.0  1109.0   2410851.0  2406548.0  2400979.0  2394478.0  2383107.0  2369234.0  2353904.0  2337449.0  2330537.0
1       id_1.10  Показатель первичной заболеваемости туберкулез...    231.15  207.52  189.40  169.03  139.92  124.90  135.12  118.99  112.70     805.0   730.0   675.0   609.0   508.0   467.0   520.0   461.0   438.0    348260.0   351781.0   356380.0   360286.0   363064.0   373903.0   384840.0   387437.0   388659.0
2     id_1.10.1  Показатель первичной заболеваемости туберкулез...    329.44  308.12  268.66  244.30  197.60  172.03  193.40  180.18  154.88     548.0   518.0   459.0   424.0   347.0   310.0   357.0   335.0   289.0    166344.0   168114.0   170847.0   173554.0   175607.0   180203.0   184591.0   185924.0   186602.0
3     id_1.10.2  Показатель первичной заболеваемости туберкулез...    141.27  115.43  116.42   99.07   85.89   81.05   81.40   62.53   73.74     257.0   212.0   216.0   185.0   161.0   157.0   163.0   126.0   149.0    181917.0   183667.0   185534.0   186732.0   187457.0   193700.0   200249.0   201513.0   202057.0
4       id_1.11  Показатель первичной заболеваемости туберкулез...    121.81  121.15   99.86   99.97   82.42   76.65   81.12   89.43   83.96     355.0   347.0   284.0   285.0   237.0   226.0   245.0   273.0   258.0    291441.0   286420.0   284386.0   285077.0   287566.0   294843.0   302033.0   305264.0   307305.0
5     id_1.11.1  Показатель первичной заболеваемости туберкулез...    186.30  185.72  147.60  154.87  120.53  118.71  124.51  146.86  134.61     251.0   246.0   194.0   204.0   160.0   161.0   172.0   204.0   188.0    134731.0   132457.0   131440.0   131723.0   132744.0   135627.0   138137.0   138910.0   139659.0
6     id_1.11.2  Показатель первичной заболеваемости туберкулез...     66.36   65.60   58.84   52.82   49.73   40.82   44.54   41.48   41.75     104.0   101.0    90.0    81.0    77.0    65.0    73.0    69.0    70.0    156710.0   153964.0   152946.0   153355.0   154822.0   159217.0   163896.0   166354.0   167646.0
7       id_1.12  Показатель первичной заболеваемости туберкулез...     80.64   74.04   61.74   55.47   45.71   50.24   49.79   42.01   39.01     257.0   235.0   194.0   171.0   137.0   148.0   143.0   116.0   106.0    318713.0   317414.0   314232.0   308254.0   299733.0   294570.0   287228.0   276137.0   271701.0
8     id_1.12.1  Показатель первичной заболеваемости туберкулез...    136.76  133.22  104.68   93.30   80.60   85.71   86.87   71.43   69.85     182.0   177.0   138.0   121.0   102.0   107.0   106.0    84.0    81.0    133076.0   132864.0   131825.0   129695.0   126549.0   124839.0   122027.0   117596.0   115967.0
9     id_1.12.2  Показатель первичной заболеваемости туберкулез...     40.40   31.43   30.70   28.00   20.21   24.16   22.40   20.18   16.05      75.0    58.0    56.0    50.0    35.0    41.0    37.0    32.0    25.0    185638.0   184550.0   182408.0   178559.0   173184.0   169731.0   165201.0   158541.0   155734.0

In [20]:
# Сохраняем результат в файл
df_pivot.to_csv(path+'result_2016-2024_широкий.csv', index=False)

In [21]:
# Сохраняем результат в файл
df_results.to_csv(path+'result_2016-2024_длинный.csv', index=False)

**Вывод:** сформирован файл с с относительными показателями, который в дальнейшем используется для построения дашборда.